# Learning Objectives
In this notebook, you will learn Spark Dataframe APIs.

# Question List

Solve the following questions using Spark Dataframe APIs

### Join

1. easy - https://pgexercises.com/questions/joins/simplejoin.html
2. easy - https://pgexercises.com/questions/joins/simplejoin2.html
3. easy - https://pgexercises.com/questions/joins/self2.html 
4. medium - https://pgexercises.com/questions/joins/threejoin.html (three join)
5. medium - https://pgexercises.com/questions/joins/sub.html (subquery and join)

### Aggregation

1. easy - https://pgexercises.com/questions/aggregates/count3.html Group by order by
2. easy - https://pgexercises.com/questions/aggregates/fachours.html group by order by
3. easy - https://pgexercises.com/questions/aggregates/fachoursbymonth.html group by with condition 
4. easy - https://pgexercises.com/questions/aggregates/fachoursbymonth2.html group by multi col
5. easy - https://pgexercises.com/questions/aggregates/members1.html count distinct
6. med - https://pgexercises.com/questions/aggregates/nbooking.html group by multiple cols, join

### String & Date

1. easy - https://pgexercises.com/questions/string/concat.html format string
2. easy - https://pgexercises.com/questions/string/case.html WHERE + string function
3. easy - https://pgexercises.com/questions/string/reg.html WHERE + string function
4. easy - https://pgexercises.com/questions/string/substr.html group by, substr
5. easy - https://pgexercises.com/questions/date/series.html generate ts
6. easy - https://pgexercises.com/questions/date/bookingspermonth.html extract month from ts

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

bookings_df = spark.read.table("bookings")
members_df = spark.read.table("members")
facilities_df = spark.read.table("facilities")

mbs = members_df.alias("mbs")
bks = bookings_df.alias("bks")
fcs = facilities_df.alias("fcs")

### Question

How can you produce a list of the start times for bookings by members named 'David Farrell'?

https://pgexercises.com/questions/joins/simplejoin.html

In [0]:
result_df = (
    bks.join(mbs, bks.memid == mbs.memid, "inner")
    .select("starttime")
    .filter((col("mbs.firstname") == "David") & (col("mbs.surname") == "Farrell"))
    .show()
)


+-------------------+
|          starttime|
+-------------------+
|2012-09-18 09:00:00|
|2012-09-18 17:30:00|
|2012-09-18 13:30:00|
|2012-09-18 20:00:00|
|2012-09-19 09:30:00|
|2012-09-19 15:00:00|
|2012-09-19 12:00:00|
|2012-09-20 15:30:00|
|2012-09-20 11:30:00|
|2012-09-20 14:00:00|
|2012-09-21 10:30:00|
|2012-09-21 14:00:00|
|2012-09-22 08:30:00|
|2012-09-22 17:00:00|
|2012-09-23 08:30:00|
|2012-09-23 17:30:00|
|2012-09-23 19:00:00|
|2012-09-24 08:00:00|
|2012-09-24 16:30:00|
|2012-09-24 12:30:00|
+-------------------+
only showing top 20 rows



### Question
How can you produce a list of the start times for bookings for tennis courts, for the date '2012-09-21'? Return a list of start time and facility name pairings, ordered by the time. 

https://pgexercises.com/questions/joins/simplejoin2.html

In [0]:
result_df = (
    bks.join(fcs, bks.memid == fcs.facid)
    .select("name","starttime")
    .filter(
        (col("name").ilike("Tennis%")) &
        (col("starttime") >= "2012-09-21") &
        (col("starttime") < "2012-09-22")
    )
    .orderBy("starttime")
    .show()
)

+--------------+-------------------+
|          name|          starttime|
+--------------+-------------------+
|Tennis Court 1|2012-09-21 09:30:00|
|Tennis Court 1|2012-09-21 10:30:00|
|Tennis Court 2|2012-09-21 11:30:00|
|Tennis Court 1|2012-09-21 11:30:00|
|Tennis Court 1|2012-09-21 13:00:00|
|Tennis Court 1|2012-09-21 14:00:00|
|Tennis Court 1|2012-09-21 14:00:00|
|Tennis Court 1|2012-09-21 15:30:00|
|Tennis Court 2|2012-09-21 17:30:00|
|Tennis Court 1|2012-09-21 17:30:00|
|Tennis Court 2|2012-09-21 19:30:00|
+--------------+-------------------+



### Question
How can you output a list of all members, including the individual who recommended them (if any)? Ensure that results are ordered by (surname, firstname). 

https://pgexercises.com/questions/joins/self2.html

In [0]:
ref = members_df.alias("ref")

result_df = (
    mbs.join(ref, col("mbs.recommendedby") == col("ref.memid"), "left_outer")
    .select(
        col("mbs.firstname").alias("memfn"),
        col("mbs.surname").alias("memsn"),
        col("ref.firstname").alias("reffn"),
        col("ref.surname").alias("refsn")
    )
  .orderBy("mbs.surname", "mbs.firstname")
  .show()
)

+---------+---------+---------+--------+
|    memfn|    memsn|    reffn|   refsn|
+---------+---------+---------+--------+
| Florence|    Bader|   Ponder|Stibbons|
|     Anne|    Baker|   Ponder|Stibbons|
|  Timothy|    Baker|   Jemima| Farrell|
|      Tim|   Boothe|      Tim|  Rownam|
|   Gerald|  Butters|   Darren|   Smith|
|     Joan|   Coplin|  Timothy|   Baker|
|    Erica|  Crumpet|    Tracy|   Smith|
|    Nancy|     Dare|   Janice|Joplette|
|    David|  Farrell|     null|    null|
|   Jemima|  Farrell|     null|    null|
|    GUEST|    GUEST|     null|    null|
|  Matthew|  Genting|   Gerald| Butters|
|     John|     Hunt|Millicent| Purview|
|    David|    Jones|   Janice|Joplette|
|  Douglas|    Jones|    David|   Jones|
|   Janice| Joplette|   Darren|   Smith|
|     Anna|Mackenzie|   Darren|   Smith|
|  Charles|     Owen|   Darren|   Smith|
|    David|   Pinker|   Jemima| Farrell|
|Millicent|  Purview|    Tracy|   Smith|
+---------+---------+---------+--------+
only showing top

### Question
How can you produce a list of all members who have used a tennis court? Include in your output the name of the court, and the name of the member formatted as a single column. Ensure no duplicate data, and order by the member name followed by the facility name. 

https://pgexercises.com/questions/joins/threejoin.html

In [0]:
result_df = (
    mbs.join(bks, col("mbs.memid") == col("bks.memid"))
    .join(fcs, col("bks.facid") == col("fcs.facid"))
    .filter(col("fcs.name").like("Tennis%"))
    .select(
        concat_ws(" ", col("mbs.firstname"), col("mbs.surname")).alias("member_name"),
        col("fcs.name").alias("court_name")
    )
    .distinct()
    .orderBy("member_name", "court_name")
    .show()
)

+--------------+--------------+
|   member_name|    court_name|
+--------------+--------------+
|    Anne Baker|Tennis Court 1|
|    Anne Baker|Tennis Court 2|
|  Burton Tracy|Tennis Court 1|
|  Burton Tracy|Tennis Court 2|
|  Charles Owen|Tennis Court 1|
|  Charles Owen|Tennis Court 2|
|  Darren Smith|Tennis Court 2|
| David Farrell|Tennis Court 1|
| David Farrell|Tennis Court 2|
|   David Jones|Tennis Court 1|
|   David Jones|Tennis Court 2|
|  David Pinker|Tennis Court 1|
| Douglas Jones|Tennis Court 1|
| Erica Crumpet|Tennis Court 1|
|Florence Bader|Tennis Court 1|
|Florence Bader|Tennis Court 2|
|   GUEST GUEST|Tennis Court 1|
|   GUEST GUEST|Tennis Court 2|
|Gerald Butters|Tennis Court 1|
|Gerald Butters|Tennis Court 2|
+--------------+--------------+
only showing top 20 rows



### Questions
How can you output a list of all members, including the individual who recommended them (if any), without using any joins? Ensure that there are no duplicates in the list, and that each firstname + surname pairing is formatted as a column and ordered. 

https://pgexercises.com/questions/joins/sub.html

In [0]:
result_df = (
    members_df.select(
        concat_ws(" ", col("firstname"), col("surname")).alias("member_name"),
        when(col("recommendedby").isNotNull(), 
         concat_ws(" ", "firstname", "surname")
         ).alias("reference_name"))
    .dropDuplicates()
    .orderBy("member_name")
    .show()
)

+--------------------+--------------------+
|         member_name|      reference_name|
+--------------------+--------------------+
|      Anna Mackenzie|      Anna Mackenzie|
|          Anne Baker|          Anne Baker|
|        Burton Tracy|                null|
|        Charles Owen|        Charles Owen|
|        Darren Smith|                null|
|       David Farrell|                null|
|         David Jones|         David Jones|
|        David Pinker|        David Pinker|
|       Douglas Jones|       Douglas Jones|
|       Erica Crumpet|       Erica Crumpet|
|      Florence Bader|      Florence Bader|
|         GUEST GUEST|                null|
|      Gerald Butters|      Gerald Butters|
|    Henrietta Rumney|    Henrietta Rumney|
|Henry Worthington...|Henry Worthington...|
| Hyacinth Tupperware|                null|
|          Jack Smith|          Jack Smith|
|     Janice Joplette|     Janice Joplette|
|      Jemima Farrell|                null|
|         Joan Coplin|         J

### Question
Produce a count of the number of recommendations each member has made. Order by member ID. 

https://pgexercises.com/questions/aggregates/count3.html

In [0]:
result_df = (
    mbs.filter(col("recommendedby").isNotNull())
    .groupBy(col("recommendedby").alias("member_id"))
    .agg(F.count("*").alias("num_references"))
    .orderBy("member_id")
    .show()
)

+---------+--------------+
|member_id|num_references|
+---------+--------------+
|        1|             5|
|        2|             3|
|        3|             1|
|        4|             2|
|        5|             1|
|        6|             1|
|        9|             2|
|       11|             1|
|       13|             2|
|       15|             1|
|       16|             1|
|       20|             1|
|       30|             1|
+---------+--------------+



### Question
Produce a list of the total number of slots booked per facility. For now, just produce an output table consisting of facility id and slots, sorted by facility id.

https://pgexercises.com/questions/aggregates/fachours.html

In [0]:
result_df = (
    bks.groupBy(col("facid").alias("facility_id"))
    .agg(F.sum("slots").alias("num_slots"))
    .orderBy("facid")
    .show()
)

+-----------+---------+
|facility_id|num_slots|
+-----------+---------+
|          0|     1320|
|          1|     1278|
|          2|     1209|
|          3|      830|
|          4|     1404|
|          5|      228|
|          6|     1104|
|          7|      908|
|          8|      911|
+-----------+---------+



### Question
Produce a list of the total number of slots booked per facility in the month of September 2012. Produce an output table consisting of facility id and slots, sorted by the number of slots. 

https://pgexercises.com/questions/aggregates/fachoursbymonth.html

In [0]:
result_df = (
    bks.filter((col("starttime") >= "2012-09-01") & (col("starttime") < "2012-10-01"))
    .groupBy("facid")
    .agg(sum("slots").alias("num_slots"))
    .orderBy(col("num_slots"))
    .show()
)

+-----+---------+
|facid|num_slots|
+-----+---------+
|    5|      122|
|    3|      422|
|    7|      426|
|    8|      471|
|    6|      540|
|    2|      570|
|    1|      588|
|    0|      591|
|    4|      648|
+-----+---------+



### Question
Produce a list of the total number of slots booked per facility per month in the year of 2012. Produce an output table consisting of facility id and slots, sorted by the id and month. 

https://pgexercises.com/questions/aggregates/fachoursbymonth2.html

In [0]:
result_df = (
    bks.filter((col("starttime") >= "2012-01-01") & (col("starttime") < "2012-12-31"))
    .withColumn("month", F.date_format("starttime", "yyyy-MM"))
    .groupBy("facid", "month")
    .agg(sum("slots").alias("num_slots"))
    .orderBy("facid", "month")
    .show()
)

+-----+-------+---------+
|facid|  month|num_slots|
+-----+-------+---------+
|    0|2012-07|      270|
|    0|2012-08|      459|
|    0|2012-09|      591|
|    1|2012-07|      207|
|    1|2012-08|      483|
|    1|2012-09|      588|
|    2|2012-07|      180|
|    2|2012-08|      459|
|    2|2012-09|      570|
|    3|2012-07|      104|
|    3|2012-08|      304|
|    3|2012-09|      422|
|    4|2012-07|      264|
|    4|2012-08|      492|
|    4|2012-09|      648|
|    5|2012-07|       24|
|    5|2012-08|       82|
|    5|2012-09|      122|
|    6|2012-07|      164|
|    6|2012-08|      400|
+-----+-------+---------+
only showing top 20 rows



### Question
Find the total number of members (including guests) who have made at least one booking.

https://pgexercises.com/questions/aggregates/members1.html

In [0]:
result_df = (
    bks.select("memid")
    .distinct()
    .agg(F.count("*").alias("total_members"))
    .show()
)

+-------------+
|total_members|
+-------------+
|           30|
+-------------+



### Question
Produce a list of each member name, id, and their first booking after September 1st 2012. Order by member ID. 

https://pgexercises.com/questions/aggregates/nbooking.html

In [0]:
result_df = (
    mbs.join(bks, col("mbs.memid") == col("bks.memid"))
    .select("mbs.firstname", "mbs.surname", "mbs.memid", "starttime")
    .filter(col("starttime") >= "2012-09-01")
    .groupBy("mbs.memid", "mbs.firstname", "mbs.surname")
    .agg(F.min("starttime").alias("first_booking"))
    .orderBy("mbs.memid")
    .show()
)

+-----+---------+---------+-------------------+
|memid|firstname|  surname|      first_booking|
+-----+---------+---------+-------------------+
|    0|    GUEST|    GUEST|2012-09-01 08:00:00|
|    1|   Darren|    Smith|2012-09-01 09:00:00|
|    2|    Tracy|    Smith|2012-09-01 11:30:00|
|    3|      Tim|   Rownam|2012-09-01 16:00:00|
|    4|   Janice| Joplette|2012-09-01 15:00:00|
|    5|   Gerald|  Butters|2012-09-02 12:30:00|
|    6|   Burton|    Tracy|2012-09-01 15:00:00|
|    7|    Nancy|     Dare|2012-09-01 12:30:00|
|    8|      Tim|   Boothe|2012-09-01 08:30:00|
|    9|   Ponder| Stibbons|2012-09-01 11:00:00|
|   10|  Charles|     Owen|2012-09-01 11:00:00|
|   11|    David|    Jones|2012-09-01 09:30:00|
|   12|     Anne|    Baker|2012-09-01 14:30:00|
|   13|   Jemima|  Farrell|2012-09-01 09:30:00|
|   14|     Jack|    Smith|2012-09-01 11:00:00|
|   15| Florence|    Bader|2012-09-01 10:30:00|
|   16|  Timothy|    Baker|2012-09-01 15:00:00|
|   17|    David|   Pinker|2012-09-01 08

### Question
Output the names of all members, formatted as 'Surname, Firstname' 

https://pgexercises.com/questions/string/concat.html

In [0]:
result_df = (
    members_df.select(
        concat_ws(", ", "surname", "firstname").alias("member_name")
    )
    .distinct()
    .orderBy("member_name")
    .show()
)

+------------------+
|       member_name|
+------------------+
|   Bader, Florence|
|       Baker, Anne|
|    Baker, Timothy|
|       Boothe, Tim|
|   Butters, Gerald|
|      Coplin, Joan|
|    Crumpet, Erica|
|       Dare, Nancy|
|    Farrell, David|
|   Farrell, Jemima|
|      GUEST, GUEST|
|  Genting, Matthew|
|        Hunt, John|
|      Jones, David|
|    Jones, Douglas|
|  Joplette, Janice|
|   Mackenzie, Anna|
|     Owen, Charles|
|     Pinker, David|
|Purview, Millicent|
+------------------+
only showing top 20 rows



### Question
Perform a case-insensitive search to find all facilities whose name begins with 'tennis'. Retrieve all columns. 

https://pgexercises.com/questions/string/case.html

In [0]:
result_df = (
    fcs.filter(col("name").ilike("Tennis%"))
    .show()
)

+-----+--------------+----------+---------+-------------+------------------+
|facid|          name|membercost|guestcost|initialoutlay|monthlymaintenance|
+-----+--------------+----------+---------+-------------+------------------+
|    0|Tennis Court 1|       5.0|     25.0|        10000|               200|
|    1|Tennis Court 2|       5.0|     25.0|         8000|               200|
+-----+--------------+----------+---------+-------------+------------------+



### Question
You've noticed that the club's member table has telephone numbers with very inconsistent formatting. You'd like to find all the telephone numbers that contain parentheses, returning the member ID and telephone number sorted by member ID. 

https://pgexercises.com/questions/string/reg.html

In [0]:
result_df = (
    mbs.select("memid", "telephone")
    .filter(col("telephone").ilike("%(%)%"))
    .show()
)

+-----+--------------+
|memid|     telephone|
+-----+--------------+
|    0|(000) 000-0000|
|    3|(844) 693-0723|
|    4|(833) 942-4710|
|    5|(844) 078-4130|
|    6|(822) 354-9973|
|    7|(833) 776-4001|
|    8|(811) 433-2547|
|    9|(833) 160-3900|
|   10|(855) 542-5251|
|   11|(844) 536-8036|
|   13|(855) 016-0163|
|   14|(822) 163-3254|
|   15|(833) 499-3527|
|   20|(811) 972-1377|
|   21|(822) 661-2898|
|   22|(822) 499-2232|
|   24|(822) 413-1470|
|   27|(822) 989-8876|
|   28|(855) 755-9876|
|   29|(855) 894-3758|
+-----+--------------+
only showing top 20 rows



### Question
You'd like to produce a count of how many members you have whose surname starts with each letter of the alphabet. Sort by the letter, and don't worry about printing out a letter if the count is 0. 

https://pgexercises.com/questions/string/substr.html

In [0]:
result_df = (
    mbs.withColumn("first_letter", F.upper(F.substring("surname", 1, 1)))
    .groupBy("first_letter")
    .agg(F.count("*").alias("total_members"))
    .orderBy("first_letter")
    .show()
)

+------------+-------------+
|first_letter|total_members|
+------------+-------------+
|           B|            5|
|           C|            2|
|           D|            1|
|           F|            2|
|           G|            2|
|           H|            1|
|           J|            3|
|           M|            1|
|           O|            1|
|           P|            2|
|           R|            2|
|           S|            6|
|           T|            2|
|           W|            1|
+------------+-------------+



### Question
Produce a list of all the dates in October 2012. They can be output as a timestamp (with time set to midnight) or a date.
 
https://pgexercises.com/questions/date/series.html

In [0]:
result_df = spark.sql("""
    SELECT explode(sequence(timestamp('2012-10-01'), timestamp('2012-10-31'), interval 1 day)) as date
""")

result_df.show()

+-------------------+
|               date|
+-------------------+
|2012-10-01 00:00:00|
|2012-10-02 00:00:00|
|2012-10-03 00:00:00|
|2012-10-04 00:00:00|
|2012-10-05 00:00:00|
|2012-10-06 00:00:00|
|2012-10-07 00:00:00|
|2012-10-08 00:00:00|
|2012-10-09 00:00:00|
|2012-10-10 00:00:00|
|2012-10-11 00:00:00|
|2012-10-12 00:00:00|
|2012-10-13 00:00:00|
|2012-10-14 00:00:00|
|2012-10-15 00:00:00|
|2012-10-16 00:00:00|
|2012-10-17 00:00:00|
|2012-10-18 00:00:00|
|2012-10-19 00:00:00|
|2012-10-20 00:00:00|
+-------------------+
only showing top 20 rows



### Question
Return a count of bookings for each month, sorted by month.

https://pgexercises.com/questions/date/bookingspermonth.html

In [0]:
result_df = (
    bks.withColumn("month", F.date_format("starttime", "yyyy-MM"))
    .groupBy("month")
    .agg(F.count("month").alias("count"))
    .orderBy("month")
    .show()
)

+-------+-----+
|  month|count|
+-------+-----+
|2012-07|  658|
|2012-08| 1472|
|2012-09| 1913|
|2013-01|    1|
+-------+-----+

